# Linear Regression — Interactive Explorers

Companion notebook for the **Simple Linear Regression** lecture.

Three hands-on widgets:

1. **Slope & Intercept Explorer** — can you beat OLS (Ordinary Least Squares.) by hand?
2. **Outlier Impact Explorer** — how much can one point move the line?
3. **Sample Size Effect** — how does `n` affect stability?

<p style="color:red; font-weight:bold;">
Before running this notebook, please install ipywidgets first:
</p>

```python
pip install ipywidgets

In [16]:
# ---- Setup (run first) ----
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interactive_output, HBox, VBox, Button, Output, Layout
from IPython.display import display
from sklearn.linear_model import LinearRegression

# Shared dataset
DATA_X = np.array([1, 2, 3, 4, 5, 6, 7], dtype=float)
DATA_Y = np.array([10, 19, 29, 46, 53, 67, 71.11])

# OLS best-fit parameters
BEST_M = 10.15
BEST_B = -0.86

# NUS-themed palette
POINT_COLOR = '#003D7C'
LINE_COLOR  = '#D62728'
POS_COLOR   = '#2CA02C'
NEG_COLOR   = '#FF7F0E'

plt.rcParams['figure.dpi'] = 110
print('Setup complete.')

Setup complete.


---
## Widget 1 — Slope & Intercept Explorer

Move `m` and `b` by hand. Watch SSE, MSE, RMSE, and R² change.
Press **🎯 Show OLS Answer** to jump to the best-fit line.

**Goal:** convince yourself you cannot beat OLS by manual tuning.

In [17]:
m_slider = widgets.FloatSlider(value=5.0, min=-5.0, max=15.0, step=0.1,
                               description='m (slope):',
                               style={'description_width': '110px'},
                               layout=Layout(width='420px'))
b_slider = widgets.FloatSlider(value=10.0, min=-20.0, max=40.0, step=0.5,
                               description='b (intercept):',
                               style={'description_width': '110px'},
                               layout=Layout(width='420px'))
ols_btn  = Button(description='🎯 Show OLS Answer', button_style='success',
                  layout=Layout(width='200px'))
out1 = Output()

# Pre-compute OLS SSE for comparison
OLS_SSE = float(np.sum((DATA_Y - (BEST_M * DATA_X + BEST_B))**2))
SST     = float(np.sum((DATA_Y - DATA_Y.mean())**2))

def _draw_widget1(m, b):
    y_hat = m * DATA_X + b
    resid = DATA_Y - y_hat
    sse = float(np.sum(resid**2))
    mse = sse / len(DATA_X)
    rmse = float(np.sqrt(mse))
    r2 = 1 - sse / SST

    fig, ax = plt.subplots(figsize=(8, 5))
    # Residual dashed lines
    for xi, yi, yh in zip(DATA_X, DATA_Y, y_hat):
        c = POS_COLOR if yi > yh else NEG_COLOR
        ax.plot([xi, xi], [yh, yi], ls='--', color=c, lw=1.4, alpha=0.8)
    ax.scatter(DATA_X, DATA_Y, s=60, color=POINT_COLOR, zorder=3, label='data')
    xs = np.linspace(0, 8, 50)
    ax.plot(xs, m*xs + b, color=LINE_COLOR, lw=2.3,
            label=f'ŷ = {m:.2f}x + {b:.2f}')

    txt = (f'SSE  = {sse:7.2f}\n'
           f'MSE  = {mse:7.2f}\n'
           f'RMSE = {rmse:7.2f}\n'
           f'R²   = {r2:7.3f}\n'
           f'\nOptimal SSE = {OLS_SSE:.2f}\n'
           f'Gap to OLS  = {sse - OLS_SSE:+.2f}')
    ax.text(0.02, 0.98, txt, transform=ax.transAxes,
            fontsize=10, family='monospace', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    ax.set_xlim(0, 8); ax.set_ylim(-10, 85)
    ax.set_xlabel('x (ad spend)'); ax.set_ylabel('y (sales)')
    ax.set_title('Manual fit vs. OLS')
    ax.legend(loc='lower right'); ax.grid(alpha=0.25)
    plt.show()

def _refresh1(change=None):
    with out1:
        out1.clear_output(wait=True)
        _draw_widget1(m_slider.value, b_slider.value)

def _snap_ols(_):
    m_slider.value = BEST_M
    b_slider.value = BEST_B

m_slider.observe(_refresh1, names='value')
b_slider.observe(_refresh1, names='value')
ols_btn.on_click(_snap_ols)

ui1 = VBox([HBox([m_slider, b_slider]), ols_btn, out1])
_refresh1()
display(ui1)

---
## Widget 2 — Outlier Impact Explorer

Add one extreme point and see how far it drags the regression line.
The left panel shows the original fit; the right shows the fit **with** the outlier included.

**Goal:** one bad observation can dominate a model. Motivates outlier detection and robust regression.

In [ ]:
ox_slider = widgets.FloatSlider(value=8.0, min=0.0, max=10.0, step=0.1,
                                description='Outlier X:',
                                style={'description_width': '90px'},
                                layout=Layout(width='380px'))
oy_slider = widgets.FloatSlider(value=20.0, min=-20.0, max=120.0, step=1.0,
                                description='Outlier Y:',
                                style={'description_width': '90px'},
                                layout=Layout(width='380px'))
include   = widgets.Checkbox(value=True, description='Include outlier')
out2 = Output()

def _fit(x, y):
    reg = LinearRegression().fit(x.reshape(-1, 1), y)
    m, b = float(reg.coef_[0]), float(reg.intercept_)
    y_hat = m * x + b
    sst = float(np.sum((y - y.mean())**2))
    sse = float(np.sum((y - y_hat)**2))
    r2 = 1 - sse/sst if sst > 0 else 0.0
    return m, b, r2

def _draw_widget2(ox, oy, include_flag):
    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    # --- Left: original ---
    m0, b0, r2_0 = _fit(DATA_X, DATA_Y)
    ax_l.scatter(DATA_X, DATA_Y, s=60, color=POINT_COLOR, zorder=3)
    xs = np.linspace(0, 10, 50)
    ax_l.plot(xs, m0*xs + b0, color=LINE_COLOR, lw=2.3,
              label=f'ŷ = {m0:.2f}x + {b0:.2f}')
    ax_l.set_title(f'Original data\nR² = {r2_0:.3f}')
    ax_l.set_xlabel('x'); ax_l.set_ylabel('y')
    ax_l.legend(loc='upper left'); ax_l.grid(alpha=0.25)
    ax_l.set_xlim(0, 10); ax_l.set_ylim(-25, 130)

    # --- Right: with outlier ---
    if include_flag:
        x_new = np.append(DATA_X, ox)
        y_new = np.append(DATA_Y, oy)
    else:
        x_new, y_new = DATA_X.copy(), DATA_Y.copy()
    m1, b1, r2_1 = _fit(x_new, y_new)

    ax_r.scatter(DATA_X, DATA_Y, s=60, color=POINT_COLOR, zorder=3, label='data')
    if include_flag:
        ax_r.scatter([ox], [oy], s=140, color='red', marker='X',
                     zorder=4, label='outlier')
    ax_r.plot(xs, m0*xs + b0, color=LINE_COLOR, lw=1.5, ls=':', alpha=0.6,
              label='original line')
    ax_r.plot(xs, m1*xs + b1, color=LINE_COLOR, lw=2.3,
              label=f'new ŷ = {m1:.2f}x + {b1:.2f}')
    ax_r.set_title(f'With outlier\nR² = {r2_1:.3f}')
    ax_r.set_xlabel('x')
    ax_r.legend(loc='upper left'); ax_r.grid(alpha=0.25)
    ax_r.set_xlim(0, 10); ax_r.set_ylim(-25, 130)

    plt.tight_layout()
    plt.show()

def _refresh2(change=None):
    with out2:
        out2.clear_output(wait=True)
        _draw_widget2(ox_slider.value, oy_slider.value, include.value)

for w in (ox_slider, oy_slider, include):
    w.observe(_refresh2, names='value')

ui2 = VBox([HBox([ox_slider, oy_slider, include]), out2])
_refresh2()
display(ui2)

---
## Widget 3 — Sample Size Effect

Generate data from the *true* relationship **y = 10·x + noise** and fit a regression line.
Increase `n` or click **Regenerate Data** to see how the fit stabilises as the sample grows.

**Goal:** more data → more reliable estimates.

In [ ]:
n_slider = widgets.IntSlider(value=7, min=3, max=50, step=1,
                             description='n (points):',
                             style={'description_width': '100px'},
                             layout=Layout(width='380px'))
regen_btn = Button(description='🔄 Regenerate Data', button_style='info',
                   layout=Layout(width='200px'))
out3 = Output()

# Hold a mutable seed so Regenerate changes the dataset
_state3 = {'seed': 42}
TRUE_LINE_COLOR = '#888888'

def _gen_data(n, seed):
    rng = np.random.default_rng(seed)
    x = rng.uniform(1, 8, size=n)
    noise = rng.normal(0, 6, size=n)
    y = 10.0 * x + noise
    return x, y

def _draw_widget3(n):
    x, y = _gen_data(n, _state3['seed'])
    m, b, r2 = _fit(x, y)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(x, y, s=55, color=POINT_COLOR, alpha=0.85, zorder=3, label='sample')
    xs = np.linspace(0, 9, 50)
    ax.plot(xs, 10.0*xs, color=TRUE_LINE_COLOR, lw=1.4, ls='--',
            label='true: y = 10x')
    ax.plot(xs, m*xs + b, color=LINE_COLOR, lw=2.3,
            label=f'fit: ŷ = {m:.2f}x + {b:.2f}')

    txt = f'n   = {n}\nR² = {r2:.3f}\nslope bias = {m-10:+.2f}'
    ax.text(0.02, 0.98, txt, transform=ax.transAxes,
            fontsize=11, family='monospace', va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))

    ax.set_xlim(0, 9); ax.set_ylim(-10, 100)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title('Sample size & fit stability')
    ax.legend(loc='lower right'); ax.grid(alpha=0.25)
    plt.show()

def _refresh3(change=None):
    with out3:
        out3.clear_output(wait=True)
        _draw_widget3(n_slider.value)

def _regen(_):
    _state3['seed'] = int(np.random.randint(0, 10**6))
    _refresh3()

n_slider.observe(_refresh3, names='value')
regen_btn.on_click(_regen)

ui3 = VBox([HBox([n_slider, regen_btn]), out3])
_refresh3()
display(ui3)

---
### Key takeaways

- **OLS is optimal** for minimising squared error — you can't beat it by tuning sliders.
- **One outlier** can swing the slope and crush R².
- **Larger samples** give more stable estimates; small samples mislead.